# Librerías

In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Cargar dataset

In [2]:
print('Cargando dataset AI4I...')
df = pd.read_csv('Dataset/ai4i2020.csv')
print('Docuemento CSV cargado.')

Cargando dataset AI4I...
Docuemento CSV cargado.


# Exploración visual de datos

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df.shape

(10000, 14)

In [ ]:
df['Machine failure'].unique()

Existen únicamente 2 estados para la máquina: funcionamiento correcto y error.

In [6]:
df["Machine failure"].describe()

count    10000.000000
mean         0.033900
std          0.180981
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: Machine failure, dtype: float64

Para poder tener una cantidad razonable de muestra para el proceso de monitoreo, se tomarán un 20% de los datos para simular la sensorización. Esto implica que se tendrán 2000 datos para suministrar al Redis stream. Además, hay que tener en cuenta que el dataset esta desbalanceada, puesto que solo el 3.39% de los datos pertenecen a un error de la fresadora.

In [11]:
df.dtypes

UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Los tipos de datos son los esperables para la tarea.

In [3]:
features = ['Air temperature [K]', 'Process temperature [K]', 
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

X = df[features]
y = df['Machine failure']

Se toman la variable dependiente y las variables independientes.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("Proporción de clases en entrenamiento (80%)")
print(y_train.value_counts(normalize=True) * 100)

print("\nProporción de clases en prueba (20%)")
print(y_test.value_counts(normalize=True) * 100)

--- Proporción de clases en Entrenamiento (80%) ---
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

--- Proporción de clases en Prueba (20%) ---
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


Las muestras tomadas para el entrenamiento y la simulación tienen proporciones similares. Para el entrenamiento se empleará la función `DecisionTreeClassifier()` de la librería `Scikit-learn`.

In [7]:
model = DecisionTreeClassifier(class_weight='balanced' ,max_depth=3, random_state=42)

En primer lugar se verificará si los datos son capaces de generalizar el problema y si el modelo es capaz de aprender usando los datos disponibles realizando una validación cruzada y empleando la métrica `F-Score`.

In [8]:
cross_val_score(model, X_train, y_train, cv=5, scoring='f1')

array([0.41284404, 0.40366972, 0.36440678, 0.42396313, 0.40174672])

La validación cruzada retorna valores similares con una media aceptable para un dataset tan desbalanceado.

In [9]:
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [ ]:
def extract_tree_rules(model, col_names):

    tree = model.tree_
    nodes_redis = {}

    def walk_node(node_id):
        is_leaf = tree.children_left[node_id] == tree.children_right[node_id]
        node_key = f"node:{node_id}"

        if is_leaf:
            values = tree.value[node_id][0]
            predicted_class = "Failure" if values[1] > values[0] else "Normal"
            nodes_redis[node_key] = {
                "tipo": "hoja",
                "resultado": predicted_class
            }
        else:
            # Es un nodo de decisión
            variable_idx = tree.feature[node_id]
            nombre_variable = col_names[variable_idx]
            umbral = tree.threshold[node_id]
            
            nodes_redis[node_key] = {
                "tipo": "decision",
                "variable": nombre_variable,
                "umbral": str(round(umbral, 2)), # Redis guarda strings
                "hijo_menor_igual": f"nodo:{tree.children_left[node_id]}",
                "hijo_mayor": f"nodo:{tree.children_right[node_id]}"
            }
            
            # Recorrer los hijos recursivamente
            walk_node(tree.children_left[node_id])
            walk_node(tree.children_right[node_id])

    # Empezamos por la raíz (nodo 0)
    walk_node(0)
    return nodes_redis

dictionary_nodes = extract_tree_rules(model, features)

print("\n--- Estructura del Árbol extraída ---")
for key, value in dictionary_nodes.items():
    print(f"{key}: {value}")
        


--- Estructura del Árbol extraída ---
node:0: {'tipo': 'decision', 'variable': 'Rotational speed [rpm]', 'umbral': '1386.5', 'hijo_menor_igual': 'nodo:1', 'hijo_mayor': 'nodo:8'}
node:1: {'tipo': 'decision', 'variable': 'Air temperature [K]', 'umbral': '301.55', 'hijo_menor_igual': 'nodo:2', 'hijo_mayor': 'nodo:5'}
node:2: {'tipo': 'decision', 'variable': 'Torque [Nm]', 'umbral': '60.05', 'hijo_menor_igual': 'nodo:3', 'hijo_mayor': 'nodo:4'}
node:3: {'tipo': 'hoja', 'resultado': 'Normal'}
node:4: {'tipo': 'hoja', 'resultado': 'Failure'}
node:5: {'tipo': 'decision', 'variable': 'Process temperature [K]', 'umbral': '312.25', 'hijo_menor_igual': 'nodo:6', 'hijo_mayor': 'nodo:7'}
node:6: {'tipo': 'hoja', 'resultado': 'Failure'}
node:7: {'tipo': 'hoja', 'resultado': 'Failure'}
node:8: {'tipo': 'decision', 'variable': 'Tool wear [min]', 'umbral': '204.5', 'hijo_menor_igual': 'nodo:9', 'hijo_mayor': 'nodo:12'}
node:9: {'tipo': 'decision', 'variable': 'Torque [Nm]', 'umbral': '15.4', 'hijo_me

In [12]:
X_test['Machine failure'] = y_test 
X_test.to_csv('datos_sensores_test.csv', index=False)
print("Datos de prueba guardados en 'datos_sensores_test.csv'")

# Nombre del archivo de salida
nombre_archivo = "tree_model.json"

# Proceso de conversión y guardado
with open(nombre_archivo, "w", encoding="utf-8") as f:
    json.dump(dictionary_nodes, f, indent=4, ensure_ascii=False)

print(f"¡Listo! El archivo '{nombre_archivo}' ha sido creado.")

Datos de prueba guardados en 'datos_sensores_test.csv'
¡Listo! El archivo 'tree_model.json' ha sido creado.
